In [1]:
WEIBO_PATH = "../data/crawler/media_crawler/weibos"

COMMENT_PATH = "../data/crawler/media_crawler/comments"

TRENDING_FILE = "../data/crawler/weibo_trending/trendings.txt"

WEIBO_SAVE_PATH = "../data/crawler/topic_weibo.parquet"

COMMENT_SAVE_PATH = "../data/crawler/topic_comment.parquet"

In [2]:
import os
import json

weibo_data = []
comment_data = []
trending_data = set()

for content_file in os.listdir(WEIBO_PATH):
    with open(os.path.join(WEIBO_PATH, content_file), "r", encoding="utf-8") as f:
        weibo_data.extend(json.load(f))

for comments_file in os.listdir(COMMENT_PATH):
    with open(os.path.join(COMMENT_PATH, comments_file), "r", encoding="utf-8") as f:
        comment_data.extend(json.load(f))

with open(TRENDING_FILE, "r", encoding="utf-8") as f:
    trending_data.update([line.strip() for line in f if line.strip()])

In [3]:
import re
from collections import defaultdict
from datetime import datetime
from typing import List, Dict

In [4]:
def extract_topics(content: str, trending_set: set) -> str:
    """从内容中提取话题标签,仅返回存在于trending_set中的话题"""
    pattern = r'#[^#]+#'
    topics = re.findall(pattern, content)
    topics = [topic.strip('#') for topic in topics]
    # 只保留存在于trending_set中的话题
    # 需要去掉两侧的#号进行匹配
    filtered_topics = [topic for topic in topics if topic in trending_set]
    return ', '.join(filtered_topics)

In [5]:
import pandas as pd

weibo_rows = []
weibo_id_set = set()

for weibo in weibo_data:
        weibo_id = int(weibo.get('note_id'))
        
        # 去除重复微博
        if weibo_id in weibo_id_set:
            continue
        weibo_id_set.add(weibo_id)

        content = weibo.get('content')
        
        date = weibo.get("create_date_time", "")

        weibo_rows.append({
            "weibo_id": weibo_id,
            "content": content,
            "create_time": weibo["create_date_time"],
            "create_time_ts": weibo["create_time"],
            "like_count": int(weibo["liked_count"]),
            "comment_count": int(weibo["comments_count"]),
            "repost_count": int(weibo["shared_count"]),
            # "ip_location": weibo["ip_location"],
            "user_id": int(weibo["user_id"]),
            "screen_name": weibo["nickname"],
            "gender": weibo["gender"],
            "topic": extract_topics(content, trending_data),
        })

df_weibo = pd.DataFrame(weibo_rows)

# 处理时间字段，将字符串类型转换为 datetime 类型
df_weibo["create_time"] = pd.to_datetime(df_weibo["create_time"])
df_weibo["create_time"] = df_weibo["create_time"].dt.tz_localize(None)

In [6]:
comment_rows = []
# child_parent_map = {}
comment_id_set = set()

for comment in comment_data:
    
    comment_id = int(comment["comment_id"])
    parent_id = int(comment["parent_comment_id"])

    parent_id = parent_id if parent_id != comment_id else -1

    if comment_id in comment_id_set:
        continue
    comment_id_set.add(comment_id)

    # if comment_id != parent_id:
    #     if parent_id in child_parent_map:
    #         print("存在多级评论")
    #     child_parent_map[comment_id] = parent_id

    date = comment["create_date_time"]
    # 构建评论字典
    comment_rows.append({
        "comment_id": comment_id,
        "parent_id": parent_id,
        "weibo_id": int(comment["note_id"]),
        "user_id": int(comment["user_id"]),
        "screen_name": comment["nickname"],
        "content": comment["content"],
        "create_time": date,
        "create_time": comment["create_time"],
        "sub_comment_count": int(comment["sub_comment_count"]),
        "like_count": int(comment["comment_like_count"]),
        "ip_location": comment["ip_location"],
        "gender": comment["gender"]
    })

df_comment = pd.DataFrame(comment_rows)
# 处理时间字段，将字符串类型转换为 datetime 类型
df_comment["create_time"] = pd.to_datetime(df_comment["create_time"])
df_comment["create_time"] = df_comment["create_time"].dt.tz_localize(None)

In [7]:
df_weibo.to_parquet(WEIBO_SAVE_PATH, index=False)
df_comment.to_parquet(COMMENT_SAVE_PATH, index=False)